In [23]:
from scipy.interpolate import BSpline

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from dataclasses import dataclass, field
from typing import List

from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

from sklearn.metrics import r2_score, mean_squared_error,mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import pickle
import os
import pandas as pd
import re
import joblib

# generate_bspline_basis_custom

In [24]:
def generate_bspline_basis_custom(knots, degree, x_norm, n_ctrl):
    B = np.zeros((len(x_norm), n_ctrl))
    for i in range(n_ctrl):
        c = np.zeros(n_ctrl)
        c[i] = 1.0
        spline = BSpline(knots, c, degree)
        B[:, i] = spline(x_norm)
    return B


In [25]:
class BsplineExpandLayerCustom(nn.Module):
    def __init__(self, knots: np.ndarray, degree: int, x_norm_points: np.ndarray):
        super().__init__()
        n_ctrl = len(knots) - degree - 1
        B_np = generate_bspline_basis_custom(knots, degree, x_norm_points, n_ctrl)
        self.register_buffer("B", torch.tensor(B_np, dtype=torch.float32))  # [23, 4]

    def forward(self, ctrl_pts):  # ctrl_pts: [B, 4]
        return torch.matmul(ctrl_pts, self.B.T)  # [B, 23]

# define the function from control_points to chord and twist for each segment

In [26]:
degree = 3
chord_knots = np.concatenate(([0]*degree, [0.3984874, 0.89904882], [1]*degree))
twist_knots = np.concatenate(([0]*degree, [0.2, 0.89904882], [1]*degree))
x_list = [0.0201254, 0.0252054, 0.0315579, 0.0379079, 0.0442579, 0.0506079, 0.0569579,
 0.0633079, 0.0696579, 0.0760079, 0.0823586, 0.0887148, 0.0950769, 0.1014424,
 0.107804,  0.1141792, 0.1173761, 0.1205864, 0.1238169, 0.1255,    0.12625,
 0.127]
x_norm = np.array(x_list) / 0.127 # the same as geometry_baseline[:,0] / 0.127


# Propeller Modeling Net

## congig for model

In [27]:
# ============================
# 配置类：定义网络参数结构
# ============================
@dataclass
class NetConfig:
    input_dim: int = 49
    hidden_dims: List[int] = field(default_factory=lambda: [128, 128])  # MLP hidden layers
    output_dim: int = 4
    use_batchnorm: bool = True

In [28]:
class PropellerPredictor(nn.Module):
    def __init__(self, input_dim=47, hidden_dims=[128, 64], output_dim=4):
        super().__init__()
        layers = []
        dims = [input_dim] + hidden_dims
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            layers.append(nn.BatchNorm1d(dims[i + 1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(dims[-1], output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [29]:
# class PropellerPredictor(nn.Module):
#     def __init__(self, config: NetConfig, chord_knots, twist_knots, x_norm):
#         super().__init__()
#         self.config = config
#         degree = len(chord_knots) - 4 - 1  # default degree is 3

#         # BSpline basis functions
#         self.chord_layer = BsplineExpandLayerCustom(chord_knots, degree, x_norm)
#         self.twist_layer = BsplineExpandLayerCustom(twist_knots, degree, x_norm)

#         # MLP structure
#         layers = []
#         prev_dim = config.input_dim
#         for hidden_dim in config.hidden_dims:
#             layers.append(nn.Linear(prev_dim, hidden_dim))
#             if config.use_batchnorm:
#                 layers.append(nn.BatchNorm1d(hidden_dim))
#             layers.append(nn.ReLU())
#             prev_dim = hidden_dim
#         layers.append(nn.Linear(prev_dim, config.output_dim))
#         self.mlp = nn.Sequential(*layers)


#     def forward(self, input):  # input shape: [B, 11]
#         rpm = input[:, 0:1]
#         aoa = input[:, 1:2]
#         wind = input[:, 2:3]
#         chord_ctrl = input[:, 3:7]
#         twist_ctrl = input[:, 7:11]

#         chord_23 = self.chord_layer(chord_ctrl)  # [B, 23]
#         twist_23 = self.twist_layer(twist_ctrl)  # [B, 23]

#         net_input = torch.cat([rpm, aoa, wind, chord_23, twist_23], dim=1)  # [B, 49]
#         return self.mlp(net_input)  # output shape: [B, 4]
# model = PropellerPredictor(config, chord_knots, twist_knots, x_norm)

# traing

## create a tensorboard for recording the training process

In [30]:
log_dir = f'runs/propeller_predictor_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
writer = SummaryWriter(log_dir=log_dir)


## deal_data for trainging LFM model

In [31]:
data_path = './data_for_train'
data_files = os.listdir(data_path)
all_data = {}
for file in data_files:
    if file.endswith('.pkl') and 'data' in str(file):
        with open(os.path.join(data_path, file), 'rb') as f:
            data = pickle.load(f)
            all_data.update(data)

### data_structure
- keys:geometry_num
  - values: 
    - keys: RMP****_Wind**_Angle**
      - values: Time,Thrust,Power,Torch,Trust_y,Thrust_z -- dataframes

In [32]:
columns_runningconditon = ['geometry_num', 'RPM', 'WIND', 'ANGLE',]
chord_columns = [f'chord_{i}' for i in range(22)]
twist_columns = [f'twist_{i}' for i in range(22)]
output_columns = ['Power', 'Fx', 'Fy', 'Fz', 'Torque']
columns = columns_runningconditon + chord_columns + twist_columns + output_columns
data_total = pd.DataFrame(columns=columns) # create an empty dataframe to store all data

# iteration all_data
for geo_idx, (key, value) in enumerate(all_data.items()):
    chord = None
    twist = None
    data_list = []

    for key_, value_ in value.items():
        # store the geometry
        if 'geometry' in key_:
            chord = value_[:, 1]
            twist = value_[:, 2]
        
        elif 'RPM' in key_:
            # store the running condition
            numbers = re.findall(r'\d+(?:\.\d+)?', key_) 
            RPM, WIND, ANGLE = map(float, numbers)
            data_one = pd.DataFrame(columns=columns)
            data_one.at[0, 'geometry_num'] = geo_idx
            data_one.at[0, 'RPM'] = RPM
            data_one.at[0, 'WIND'] = WIND
            data_one.at[0, 'ANGLE'] = ANGLE
            # store the output data
            data_one.at[0, 'Fx'] = -value_['Thrust'].mean() # (because of the direction of thrust)
            data_one.at[0, 'Fy'] = value_['Trust_y'].mean()
            data_one.at[0, 'Fz'] = value_['Trust_z'].mean()
            data_one.at[0, 'Power'] = -value_['Power'].mean() # (because of the direction of thrust)
            data_one.at[0, 'Torque'] = -value_['Torque'].mean() # (because of the direction of thrust)

            # ignore the chord and twist data because the order is not sure
            data_list.append(data_one)

    # geometry data has been stored, now store the chord and twist data 
    if chord is not None and twist is not None:
        for data_one in data_list:
            for i in range(22):
                data_one.at[0, f'chord_{i}'] = chord[i]
                data_one.at[0, f'twist_{i}'] = twist[i]
            data_total = pd.concat([data_total, data_one], ignore_index=True)
# save the data
data_path = os.path.join('./data_for_train', 'dealed_data.xlsx')
data_total.to_excel(data_path)

#### standard the data

In [33]:
# load the data from excel
df = pd.read_excel('./data_for_train/dealed_data.xlsx')

# define the columns we need
columns_runningconditon = ['RPM', 'WIND', 'ANGLE',]
chord_columns = [f'chord_{i}' for i in range(22)]
twist_columns = [f'twist_{i}' for i in range(22)]
output_columns = ['Fx', 'Fy', 'Fz', 'Torque']

# split the data into Input set and Output set
X = df[columns_runningconditon + chord_columns + twist_columns]
Y = df[output_columns]
# split the data into training set and testing set
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# initialize the scaler
scaler_runningconditon = MinMaxScaler()
scaler_chord = MinMaxScaler()
scaler_twist = MinMaxScaler()

# fit the scaler on the training set
X_train_runningconditon_scaled = scaler_runningconditon.fit_transform(X_train[columns_runningconditon])
X_train_chord_scaled = scaler_chord.fit_transform(X_train[chord_columns]) 
X_train_twist_scaled = scaler_twist.fit_transform(X_train[twist_columns]) 

# transform the testing set
X_test_runningconditon_scaled = scaler_runningconditon.transform(X_test[columns_runningconditon])
X_test_chord_scaled = scaler_chord.transform(X_test[chord_columns]) 
X_test_twist_scaled = scaler_twist.transform(X_test[twist_columns]) 

# Results after merger normalization
X_train_scaled = np.hstack([X_train_runningconditon_scaled, X_train_chord_scaled, X_train_twist_scaled])
X_test_scaled = np.hstack([X_test_runningconditon_scaled, X_test_chord_scaled, X_test_twist_scaled])

# transform into dataframe
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=columns_runningconditon + chord_columns + twist_columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=columns_runningconditon + chord_columns + twist_columns)

# save the data into pickle file
X_train_scaled_df.to_pickle(os.path.join('./data_for_train', 'X_train_scaled'))
X_test_scaled_df.to_pickle(os.path.join('./data_for_train', 'X_test_scaled'))
y_train.to_pickle(os.path.join('./data_for_train', 'y_train'))
y_test.to_pickle(os.path.join('./data_for_train', 'y_test'))

# save the scaler
joblib.dump(scaler_runningconditon, os.path.join('./data_for_train', 'runningconditon_scaler.pkl'))
joblib.dump(scaler_chord, os.path.join('./data_for_train', 'chord_scaler.pkl'))
joblib.dump(scaler_twist, os.path.join('./data_for_train', 'twist_scaler.pkl'))

['./data_for_train\\twist_scaler.pkl']

## Hyperparameters

In [34]:
config = NetConfig(
    input_dim=47,
    hidden_dims=[128, 64, 32],  # three hidden layers with 128, 64, 32 neurons respectively
    output_dim=4,
    use_batchnorm=True,
)
epochs = 100

# define the model
config = NetConfig(hidden_dims=[128, 64, 32])
device = torch.device("cpu")
model = PropellerPredictor().to(device)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
batch_size = 64
epochs = 5000


## traing

### eval for tensorboard recording

In [35]:
def evaluate_metrics(y_true, y_pred):
    y_true_np = y_true.detach().cpu().numpy()
    y_pred_np = y_pred.detach().cpu().numpy()
    mae = mean_absolute_error(y_true_np, y_pred_np)
    r2 = r2_score(y_true_np, y_pred_np)
    rel_error = np.abs((y_pred_np - y_true_np) / (np.abs(y_true_np) + 1e-8))
    return mae, r2, rel_error

### load data

In [36]:
X_train = torch.tensor(pd.read_pickle('./data_for_train/X_train_scaled').values, dtype=torch.float32)
X_test = torch.tensor(pd.read_pickle('./data_for_train/X_test_scaled').values, dtype=torch.float32)
y_train = torch.tensor(pd.read_pickle('./data_for_train/y_train').values, dtype=torch.float32)
y_test = torch.tensor(pd.read_pickle('./data_for_train/y_test').values, dtype=torch.float32)

# dataloader setting
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)

In [37]:
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = loss_fn(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_X.size(0)
    model.eval()
    with torch.no_grad():
        train_outputs = model(X_train)
        test_outputs = model(X_test)

        # record loss
        writer.add_scalar("Loss/Train", loss_fn(train_outputs, y_train).item(), epoch)
        writer.add_scalar("Loss/Test", loss_fn(test_outputs, y_test).item(), epoch)

        # record MAE, R², relative error
        train_mae, train_r2, train_rel = evaluate_metrics(y_train, train_outputs)
        test_mae, test_r2, test_rel = evaluate_metrics(y_test, test_outputs)
        writer.add_scalar("MAE/Train", train_mae, epoch)
        writer.add_scalar("MAE/Test", test_mae, epoch)
        writer.add_scalar("R2/Train", train_r2, epoch)
        writer.add_scalar("R2/Test", test_r2, epoch)
        writer.add_scalar("RelativeError/Train", train_rel.mean(), epoch)
        writer.add_scalar("RelativeError/Test", test_rel.mean(), epoch)
        output_names = ['Power', 'Fx', 'Fy', 'Fz']
        for i, name in enumerate(output_names):
            writer.add_scalar(f'RelativeError/Train/{name}', np.mean(train_rel[:, i]), epoch)
            writer.add_scalar(f'RelativeError/Test/{name}', np.mean(test_rel[:, i]), epoch)

    avg_loss = total_loss / len(train_loader.dataset)
    writer.add_scalar("Loss/train", avg_loss, epoch)
    
    # print information for each 500
    if epoch % 500 == 0:
        print(f"[Epoch {epoch}] Loss: {avg_loss:.6f}")
    

[Epoch 0] Loss: 3.696316
[Epoch 500] Loss: 0.001355
[Epoch 1000] Loss: 0.000773
[Epoch 1500] Loss: 0.000477
[Epoch 2000] Loss: 0.000313
[Epoch 2500] Loss: 0.000356
[Epoch 3000] Loss: 0.000297
[Epoch 3500] Loss: 0.000256
[Epoch 4000] Loss: 0.000346
[Epoch 4500] Loss: 0.000235


In [38]:
## save the model to disk

In [39]:
os.makedirs("trained_models", exist_ok=True)
torch.save(model.state_dict(), "trained_models/propeller_predictor_cpu.pth")
writer.close()

# prediction

In [40]:
X_test = pd.read_pickle('./data_for_train/X_test_scaled')
y_test = pd.read_pickle('./data_for_train/y_test')
model.load_state_dict(torch.load('trained_models/propeller_predictor_cpu.pth'))  # 请根据你保存的模型文件名修改
model.eval()

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32).to(device)
with torch.no_grad():
    y_pred_tensor = model(X_test_tensor).cpu()

y_pred = y_pred_tensor.numpy()
df_result = pd.DataFrame(y_pred, columns=[ 'Fx_pred', 'Fy_pred', 'Fz_pred', 'Torque_pred'])
df_true = y_test.reset_index(drop=True)
df_all = pd.concat([df_true, df_result], axis=1)
df_all.to_excel('./data_for_train/predict_vs_true.xlsx', index=False)
